# PCG-to-ECG Signal Reconstruction: Model Architectures

**Ablation Study Code** for: *PCG-to-ECG Signal Reconstruction Using Conditional GAN with Multi-Scale Attention* (EUSIPCO 2026)

**Authors:** Zafar Ahmad, Nazia Perwaiz, Muhammad Salman Khan

---

This notebook contains all four generator architectures tested across the ablation study (Trails 1–4), plus the two discriminator variants.

| Trail | Architecture | Output Activation | Skip Connections | Params |
|-------|-------------|-------------------|-----------------|--------|
| 1 | CNN Encoder-Decoder | Tanh | No | 3.8 M |
| 2 | U-Net + MHSA + MSDC | Tanh | Yes | 19.3 M |
| 3 | U-Net + MHSA + MSDC | BatchNorm | Yes | 19.3 M |
| 4 | U-Net + MHSA + MSDC + Polarity Fix | BatchNorm | Yes | 19.3 M |

## Imports

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

## Supporting Modules

These modules are shared across Trails 2–4:
- **ResidualBlock**: Standard residual block (Trail-1 bottleneck)
- **MultiHeadSelfAttention**: 8-head attention at bottleneck (2000 timesteps, 512 channels)
- **MultiScaleDilatedBlock**: Parallel dilated convolutions at rates 1, 2, 4
- **EnhancedResidualBlock**: Combines standard conv + optional MHSA + optional MSDC

In [ ]:
class ResidualBlock(nn.Module):
    """Standard residual block (Trail-1 bottleneck)."""
    def __init__(self, channels):
        super().__init__()
        self.conv1 = nn.Conv1d(channels, channels, kernel_size=3, padding=1)
        self.bn1 = nn.BatchNorm1d(channels)
        self.conv2 = nn.Conv1d(channels, channels, kernel_size=3, padding=1)
        self.bn2 = nn.BatchNorm1d(channels)
        self.relu = nn.ReLU(inplace=True)

    def forward(self, x):
        residual = x
        out = self.relu(self.bn1(self.conv1(x)))
        out = self.bn2(self.conv2(out))
        out += residual
        return self.relu(out)

In [ ]:
class MultiHeadSelfAttention(nn.Module):
    """
    Multi-head self-attention for 1D temporal signals.
    8 heads, applied at the bottleneck (2000 timesteps, 512 channels).
    Lets the model attend to distant cardiac events (e.g. P-wave
    preceding its associated R-peak ~160 ms later).
    """
    def __init__(self, channels, num_heads=8, dropout=0.1):
        super().__init__()
        assert channels % num_heads == 0
        self.num_heads = num_heads
        self.head_dim = channels // num_heads
        self.scale = self.head_dim ** -0.5

        self.qkv = nn.Conv1d(channels, channels * 3, kernel_size=1, bias=False)
        self.proj = nn.Conv1d(channels, channels, kernel_size=1)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        B, C, T = x.shape
        qkv = self.qkv(x).reshape(B, 3, self.num_heads, self.head_dim, T)
        qkv = qkv.permute(1, 0, 2, 4, 3)  # (3, B, heads, T, head_dim)
        q, k, v = qkv[0], qkv[1], qkv[2]

        attn = (q @ k.transpose(-2, -1)) * self.scale
        attn = F.softmax(attn, dim=-1)
        attn = self.dropout(attn)

        out = (attn @ v).transpose(2, 3).reshape(B, C, T)
        out = self.proj(out)
        return out + x  # residual

In [ ]:
class MultiScaleDilatedBlock(nn.Module):
    """
    Parallel dilated convolutions at rates 1, 2, 4.
    Receptive fields at input resolution (8 kHz):
        dilation 1 -> 3 samples  = 0.375 ms  (QRS detail)
        dilation 2 -> 5 samples  = 0.625 ms  (QRS envelope)
        dilation 4 -> 9 samples  = 1.125 ms  (ST segment context)

    Channel split: ceil(C/3) + ceil(C/3) + floor(C/3) = C.
    For C=512: 171 + 171 + 170 = 512.
    """
    def __init__(self, channels):
        super().__init__()
        c1 = channels // 3 + (1 if channels % 3 >= 1 else 0)
        c2 = channels // 3 + (1 if channels % 3 >= 2 else 0)
        c3 = channels // 3

        self.conv_d1 = nn.Conv1d(channels, c1, kernel_size=3, padding=1, dilation=1)
        self.conv_d2 = nn.Conv1d(channels, c2, kernel_size=3, padding=2, dilation=2)
        self.conv_d4 = nn.Conv1d(channels, c3, kernel_size=3, padding=4, dilation=4)

        self.fusion = nn.Sequential(
            nn.Conv1d(channels, channels, kernel_size=1),
            nn.BatchNorm1d(channels),
            nn.ReLU(inplace=True),
        )

    def forward(self, x):
        d1 = self.conv_d1(x)
        d2 = self.conv_d2(x)
        d4 = self.conv_d4(x)
        return self.fusion(torch.cat([d1, d2, d4], dim=1)) + x

In [ ]:
class EnhancedResidualBlock(nn.Module):
    """Residual block with optional MHSA and MSDC (Trails 2-4)."""
    def __init__(self, channels, use_attention=False, use_dilated=True):
        super().__init__()
        self.conv1 = nn.Conv1d(channels, channels, kernel_size=3, padding=1)
        self.bn1 = nn.BatchNorm1d(channels)
        self.use_dilated = use_dilated
        if use_dilated:
            self.dilated_block = MultiScaleDilatedBlock(channels)
        self.conv2 = nn.Conv1d(channels, channels, kernel_size=3, padding=1)
        self.bn2 = nn.BatchNorm1d(channels)
        self.use_attention = use_attention
        if use_attention:
            self.attention = MultiHeadSelfAttention(channels, num_heads=8)
        self.relu = nn.ReLU(inplace=True)

    def forward(self, x):
        residual = x
        out = self.relu(self.bn1(self.conv1(x)))
        if self.use_dilated:
            out = self.dilated_block(out)
        out = self.bn2(self.conv2(out))
        if self.use_attention:
            out = self.attention(out)
        return self.relu(out + residual)

---
## Trail-1: CNN Encoder-Decoder Generator (Baseline)

- 5-level encoder-decoder with stride-2 convolutions
- 4 residual blocks in the bottleneck
- **NO** skip connections
- **Tanh** output activation (clips to [-1, +1])
- ~3.8 M parameters

**Limitation:** Tanh clips R-peaks whose z-score amplitude exceeds 1.0.

In [ ]:
class CNNEncoderDecoder(nn.Module):
    """Trail-1 baseline generator."""
    def __init__(self, signal_length=64000, base_channels=64):
        super().__init__()
        bc = base_channels

        # Encoder
        self.enc1 = nn.Sequential(
            nn.Conv1d(1, bc, 7, stride=2, padding=3),
            nn.BatchNorm1d(bc), nn.ReLU(True))
        self.enc2 = nn.Sequential(
            nn.Conv1d(bc, bc*2, 5, stride=2, padding=2),
            nn.BatchNorm1d(bc*2), nn.ReLU(True))
        self.enc3 = nn.Sequential(
            nn.Conv1d(bc*2, bc*4, 5, stride=2, padding=2),
            nn.BatchNorm1d(bc*4), nn.ReLU(True))
        self.enc4 = nn.Sequential(
            nn.Conv1d(bc*4, bc*8, 5, stride=2, padding=2),
            nn.BatchNorm1d(bc*8), nn.ReLU(True))
        self.enc5 = nn.Sequential(
            nn.Conv1d(bc*8, bc*8, 5, stride=2, padding=2),
            nn.BatchNorm1d(bc*8), nn.ReLU(True))

        # Bottleneck
        self.bottleneck = nn.Sequential(
            ResidualBlock(bc*8), ResidualBlock(bc*8),
            ResidualBlock(bc*8), ResidualBlock(bc*8))

        # Decoder
        self.dec5 = nn.Sequential(
            nn.ConvTranspose1d(bc*8, bc*8, 4, stride=2, padding=1),
            nn.BatchNorm1d(bc*8), nn.ReLU(True))
        self.dec4 = nn.Sequential(
            nn.ConvTranspose1d(bc*8, bc*4, 4, stride=2, padding=1),
            nn.BatchNorm1d(bc*4), nn.ReLU(True))
        self.dec3 = nn.Sequential(
            nn.ConvTranspose1d(bc*4, bc*2, 4, stride=2, padding=1),
            nn.BatchNorm1d(bc*2), nn.ReLU(True))
        self.dec2 = nn.Sequential(
            nn.ConvTranspose1d(bc*2, bc, 4, stride=2, padding=1),
            nn.BatchNorm1d(bc), nn.ReLU(True))
        self.dec1 = nn.Sequential(
            nn.ConvTranspose1d(bc, 1, 4, stride=2, padding=1),
            nn.Tanh())  # <-- clips output to [-1, +1]

    def forward(self, x):
        e1 = self.enc1(x)
        e2 = self.enc2(e1)
        e3 = self.enc3(e2)
        e4 = self.enc4(e3)
        e5 = self.enc5(e4)
        b = self.bottleneck(e5)
        d5 = self.dec5(b)
        d4 = self.dec4(d5)
        d3 = self.dec3(d4)
        d2 = self.dec2(d3)
        return self.dec1(d2)

---
## Trail-2: U-Net + MHSA + MSDC Generator (with Tanh)

- 5-level U-Net with skip connections
- 4 Enhanced Residual Blocks (MHSA + MSDC) in bottleneck
- Scenario conditioning (rest / exercise / recovery)
- **Tanh** output activation (still clips R-peaks)
- ~19.3 M parameters

In [ ]:
class UNetGenerator(nn.Module):
    """Trail-2 generator: U-Net + MHSA + MSDC, Tanh output."""
    def __init__(self, signal_length=64000, base_channels=64, num_scenarios=3):
        super().__init__()
        bc = base_channels
        self.scenario_embed = nn.Linear(num_scenarios, bc)

        # Encoder
        self.enc1 = nn.Sequential(
            nn.Conv1d(1, bc, 7, stride=2, padding=3),
            nn.BatchNorm1d(bc), nn.ReLU(True))
        self.enc2 = nn.Sequential(
            nn.Conv1d(bc, bc*2, 5, stride=2, padding=2),
            nn.BatchNorm1d(bc*2), nn.ReLU(True))
        self.enc3 = nn.Sequential(
            nn.Conv1d(bc*2, bc*4, 5, stride=2, padding=2),
            nn.BatchNorm1d(bc*4), nn.ReLU(True))
        self.enc4 = nn.Sequential(
            nn.Conv1d(bc*4, bc*8, 5, stride=2, padding=2),
            nn.BatchNorm1d(bc*8), nn.ReLU(True))
        self.enc5 = nn.Sequential(
            nn.Conv1d(bc*8, bc*8, 5, stride=2, padding=2),
            nn.BatchNorm1d(bc*8), nn.ReLU(True))

        # Bottleneck: 4 blocks with MHSA + MSDC
        self.bottleneck = nn.Sequential(
            EnhancedResidualBlock(bc*8, use_attention=True, use_dilated=True),
            EnhancedResidualBlock(bc*8, use_attention=True, use_dilated=True),
            EnhancedResidualBlock(bc*8, use_attention=True, use_dilated=True),
            EnhancedResidualBlock(bc*8, use_attention=True, use_dilated=True))

        # Decoder with skip connections
        self.dec5 = nn.Sequential(
            nn.ConvTranspose1d(bc*8, bc*8, 4, stride=2, padding=1),
            nn.BatchNorm1d(bc*8), nn.ReLU(True))
        self.dec4 = nn.Sequential(
            nn.Conv1d(bc*16, bc*8, 1),
            nn.ConvTranspose1d(bc*8, bc*4, 4, stride=2, padding=1),
            nn.BatchNorm1d(bc*4), nn.ReLU(True))
        self.dec3 = nn.Sequential(
            nn.Conv1d(bc*8, bc*4, 1),
            nn.ConvTranspose1d(bc*4, bc*2, 4, stride=2, padding=1),
            nn.BatchNorm1d(bc*2), nn.ReLU(True))
        self.dec2 = nn.Sequential(
            nn.Conv1d(bc*4, bc*2, 1),
            nn.ConvTranspose1d(bc*2, bc, 4, stride=2, padding=1),
            nn.BatchNorm1d(bc), nn.ReLU(True))
        self.dec1 = nn.Sequential(
            nn.Conv1d(bc*2, bc, 1),
            nn.ConvTranspose1d(bc, 1, 4, stride=2, padding=1),
            nn.Tanh())  # <-- still clips R-peaks

    def forward(self, x, scenario_onehot):
        s = self.scenario_embed(scenario_onehot).unsqueeze(2)

        e1 = self.enc1(x) + s
        e2 = self.enc2(e1)
        e3 = self.enc3(e2)
        e4 = self.enc4(e3)
        e5 = self.enc5(e4)

        b = self.bottleneck(e5)

        d5 = self.dec5(b)
        d5 = torch.cat([d5, e4], dim=1)
        d4 = self.dec4(d5)
        d4 = torch.cat([d4, e3], dim=1)
        d3 = self.dec3(d4)
        d3 = torch.cat([d3, e2], dim=1)
        d2 = self.dec2(d3)
        d2 = torch.cat([d2, e1], dim=1)
        return self.dec1(d2)

---
## Trail-3 / Trail-4: U-Net + MHSA + MSDC Generator (NO Tanh)

Same as `UNetGenerator` but with **Tanh replaced by BatchNorm1d(1)**.

BatchNorm stabilizes the output without bounding it. Its gradient is constant (~gamma / sqrt(var + eps)), unlike Tanh whose gradient drops to 0.010 at |x| = 3.0.

- **Trail-3** trained this for only 16 epochs (PSNR dropped to 14 dB)
- **Trail-4** trained for 100 epochs with polarity correction → PSNR = 17.10 dB, correlation = 0.610

In [ ]:
class ImprovedGeneratorNoTanh(nn.Module):
    """Trail-3/4 generator: Tanh replaced with BatchNorm."""
    def __init__(self, signal_length=64000, base_channels=64, num_scenarios=3):
        super().__init__()
        bc = base_channels
        self.scenario_embed = nn.Linear(num_scenarios, bc)

        # Encoder (identical to UNetGenerator)
        self.enc1 = nn.Sequential(
            nn.Conv1d(1, bc, 7, stride=2, padding=3),
            nn.BatchNorm1d(bc), nn.ReLU(True))
        self.enc2 = nn.Sequential(
            nn.Conv1d(bc, bc*2, 5, stride=2, padding=2),
            nn.BatchNorm1d(bc*2), nn.ReLU(True))
        self.enc3 = nn.Sequential(
            nn.Conv1d(bc*2, bc*4, 5, stride=2, padding=2),
            nn.BatchNorm1d(bc*4), nn.ReLU(True))
        self.enc4 = nn.Sequential(
            nn.Conv1d(bc*4, bc*8, 5, stride=2, padding=2),
            nn.BatchNorm1d(bc*8), nn.ReLU(True))
        self.enc5 = nn.Sequential(
            nn.Conv1d(bc*8, bc*8, 5, stride=2, padding=2),
            nn.BatchNorm1d(bc*8), nn.ReLU(True))

        # Bottleneck (identical)
        self.bottleneck = nn.Sequential(
            EnhancedResidualBlock(bc*8, use_attention=True, use_dilated=True),
            EnhancedResidualBlock(bc*8, use_attention=True, use_dilated=True),
            EnhancedResidualBlock(bc*8, use_attention=True, use_dilated=True),
            EnhancedResidualBlock(bc*8, use_attention=True, use_dilated=True))

        # Decoder (identical except final layer)
        self.dec5 = nn.Sequential(
            nn.ConvTranspose1d(bc*8, bc*8, 4, stride=2, padding=1),
            nn.BatchNorm1d(bc*8), nn.ReLU(True))
        self.dec4 = nn.Sequential(
            nn.Conv1d(bc*16, bc*8, 1),
            nn.ConvTranspose1d(bc*8, bc*4, 4, stride=2, padding=1),
            nn.BatchNorm1d(bc*4), nn.ReLU(True))
        self.dec3 = nn.Sequential(
            nn.Conv1d(bc*8, bc*4, 1),
            nn.ConvTranspose1d(bc*4, bc*2, 4, stride=2, padding=1),
            nn.BatchNorm1d(bc*2), nn.ReLU(True))
        self.dec2 = nn.Sequential(
            nn.Conv1d(bc*4, bc*2, 1),
            nn.ConvTranspose1d(bc*2, bc, 4, stride=2, padding=1),
            nn.BatchNorm1d(bc), nn.ReLU(True))

        # CHANGED: BatchNorm replaces Tanh
        self.dec1 = nn.Sequential(
            nn.Conv1d(bc*2, bc, 1),
            nn.ConvTranspose1d(bc, 1, 4, stride=2, padding=1),
            nn.BatchNorm1d(1))  # unbounded output

    def forward(self, x, scenario_onehot):
        s = self.scenario_embed(scenario_onehot).unsqueeze(2)

        e1 = self.enc1(x) + s
        e2 = self.enc2(e1)
        e3 = self.enc3(e2)
        e4 = self.enc4(e3)
        e5 = self.enc5(e4)

        b = self.bottleneck(e5)

        d5 = self.dec5(b)
        d5 = torch.cat([d5, e4], dim=1)
        d4 = self.dec4(d5)
        d4 = torch.cat([d4, e3], dim=1)
        d3 = self.dec3(d4)
        d3 = torch.cat([d3, e2], dim=1)
        d2 = self.dec2(d3)
        d2 = torch.cat([d2, e1], dim=1)
        return self.dec1(d2)

---
## Discriminators

In [ ]:
class BasicDiscriminator(nn.Module):
    """Trail-1 discriminator. Inputs: concatenated PCG + ECG (2 ch)."""
    def __init__(self, signal_length=64000, base_channels=64):
        super().__init__()
        bc = base_channels
        self.model = nn.Sequential(
            nn.Conv1d(2, bc, 7, stride=2, padding=3),
            nn.LeakyReLU(0.2, True),
            nn.Conv1d(bc, bc*2, 5, stride=2, padding=2),
            nn.BatchNorm1d(bc*2), nn.LeakyReLU(0.2, True),
            nn.Conv1d(bc*2, bc*4, 5, stride=2, padding=2),
            nn.BatchNorm1d(bc*4), nn.LeakyReLU(0.2, True),
            nn.Conv1d(bc*4, bc*8, 5, stride=2, padding=2),
            nn.BatchNorm1d(bc*8), nn.LeakyReLU(0.2, True),
            nn.Conv1d(bc*8, bc*8, 5, stride=2, padding=2),
            nn.BatchNorm1d(bc*8), nn.LeakyReLU(0.2, True),
            nn.AdaptiveAvgPool1d(1), nn.Flatten(),
            nn.Linear(bc*8, 1), nn.Sigmoid())

    def forward(self, pcg, ecg):
        return self.model(torch.cat([pcg, ecg], dim=1))

In [ ]:
class SpectralNormDiscriminator(nn.Module):
    """
    Trail-2/3/4 discriminator with spectral normalization
    and scenario conditioning.
    """
    def __init__(self, signal_length=64000, base_channels=64, num_scenarios=3):
        super().__init__()
        bc = base_channels
        self.scenario_embed = nn.Linear(num_scenarios, bc)

        self.conv1 = nn.utils.spectral_norm(
            nn.Conv1d(2, bc, 7, stride=2, padding=3))
        self.conv2 = nn.utils.spectral_norm(
            nn.Conv1d(bc, bc*2, 5, stride=2, padding=2))
        self.bn2 = nn.BatchNorm1d(bc*2)
        self.conv3 = nn.utils.spectral_norm(
            nn.Conv1d(bc*2, bc*4, 5, stride=2, padding=2))
        self.bn3 = nn.BatchNorm1d(bc*4)
        self.conv4 = nn.utils.spectral_norm(
            nn.Conv1d(bc*4, bc*8, 5, stride=2, padding=2))
        self.bn4 = nn.BatchNorm1d(bc*8)
        self.conv5 = nn.utils.spectral_norm(
            nn.Conv1d(bc*8, bc*8, 5, stride=2, padding=2))
        self.bn5 = nn.BatchNorm1d(bc*8)

        self.pool = nn.AdaptiveAvgPool1d(1)
        self.fc = nn.Linear(bc*8, 1)
        self.lrelu = nn.LeakyReLU(0.2, True)

    def forward(self, pcg, ecg, scenario_onehot):
        x = torch.cat([pcg, ecg], dim=1)
        s = self.scenario_embed(scenario_onehot).unsqueeze(2)

        x = self.lrelu(self.conv1(x)) + s
        x = self.lrelu(self.bn2(self.conv2(x)))
        x = self.lrelu(self.bn3(self.conv3(x)))
        x = self.lrelu(self.bn4(self.conv4(x)))
        x = self.lrelu(self.bn5(self.conv5(x)))
        x = self.pool(x).squeeze(2)
        return torch.sigmoid(self.fc(x))

---
## Parameter Count Verification

In [ ]:
def count_params(model):
    """Return total trainable parameter count."""
    return sum(p.numel() for p in model.parameters() if p.requires_grad)

for name, cls, args in [
    ("Trail-1 CNN", CNNEncoderDecoder, {}),
    ("Trail-2 UNet+Tanh", UNetGenerator, {}),
    ("Trail-3/4 UNet-NoTanh", ImprovedGeneratorNoTanh, {}),
    ("Trail-1 Discriminator", BasicDiscriminator, {}),
    ("Trail-2+ Discriminator", SpectralNormDiscriminator, {}),
]:
    m = cls(**args)
    print(f"{name:30s}  params = {count_params(m):>12,}")